# Intelligent Air Quality Forecasting and Pollution Risk Analysis Using Machine Learning
## Premium Project Notebook  
### Manushi Paudel

This notebook builds a complete machine learning workflow to analyze and forecast air pollution using the **Beijing PM2.5 dataset**. It includes:

- data cleaning and preprocessing  
- exploratory data analysis  
- feature engineering  
- **regression** for PM2.5 prediction  
- **classification** for pollution risk analysis  
- model comparison and interpretation  
- final conclusions and future improvements


## 1. Problem Statement

Air pollution has major environmental and health impacts, especially in urban areas. Among air pollutants, **PM2.5** is one of the most dangerous because fine particles can enter the lungs and bloodstream.

The goal of this project is to:

1. **Predict PM2.5 concentration** using meteorological features  
2. **Classify pollution risk levels** based on PM2.5 thresholds  
3. Understand how factors such as temperature, pressure, dew point, and wind affect air quality


## 2. Dataset

**Dataset:** Beijing PM2.5 Dataset  
**Expected file name:** `PRSA_data_2010.1.1-2014.12.31.csv`

If your CSV has a different name, change the `DATA_PATH` variable in the next cell.


In [ ]:
# 3. Import Libraries

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)


In [ ]:
# 4. Load Dataset

DATA_PATH = "PRSA_data_2010.1.1-2014.12.31.csv"

df = pd.read_csv(DATA_PATH)
print("Dataset shape:", df.shape)
df.head()


## 3. Initial Data Inspection

We first inspect the structure of the dataset, column names, missing values, and data types.


In [ ]:
df.info()


In [ ]:
df.describe(include="all").T


In [ ]:
missing = df.isnull().sum().sort_values(ascending=False).to_frame("Missing Values")
missing["Missing Percentage"] = (missing["Missing Values"] / len(df) * 100).round(2)
missing


## 4. Data Cleaning and Preprocessing

In this section:

- create a proper datetime column  
- remove unnecessary index columns if present  
- handle missing target values  
- keep both numerical and categorical predictors  
- engineer useful time-based features


In [ ]:
# Drop unnecessary index column if present
if "No" in df.columns:
    df = df.drop(columns=["No"])

# Create datetime column if date parts exist
date_parts = ["year", "month", "day", "hour"]
if all(col in df.columns for col in date_parts):
    df["date"] = pd.to_datetime(df[date_parts])

# Remove rows where target is missing
df = df[df["pm2.5"].notna()].copy()

# Feature engineering
if "month" in df.columns:
    df["season"] = df["month"] % 12 // 3 + 1
    df["season_name"] = df["season"].map({1: "Winter", 2: "Spring", 3: "Summer", 4: "Autumn"})

if "hour" in df.columns:
    df["time_of_day"] = pd.cut(
        df["hour"],
        bins=[-1, 5, 11, 17, 23],
        labels=["Night", "Morning", "Afternoon", "Evening"]
    )

print("Cleaned dataset shape:", df.shape)
df.head()


## 5. Exploratory Data Analysis

EDA helps reveal the distribution of PM2.5 and its relationship with environmental features.


In [ ]:
# PM2.5 distribution
plt.figure(figsize=(10, 5))
sns.histplot(df["pm2.5"], bins=50, kde=True)
plt.title("Distribution of PM2.5 Concentration")
plt.xlabel("PM2.5")
plt.ylabel("Frequency")
plt.show()


In [ ]:
# Boxplot for PM2.5
plt.figure(figsize=(10, 4))
sns.boxplot(x=df["pm2.5"])
plt.title("Boxplot of PM2.5 Concentration")
plt.xlabel("PM2.5")
plt.show()


In [ ]:
# PM2.5 trend over time
if "date" in df.columns:
    daily_pm = df.set_index("date")["pm2.5"].resample("D").mean()
    plt.figure(figsize=(14, 5))
    plt.plot(daily_pm.index, daily_pm.values)
    plt.title("Daily Average PM2.5 Over Time")
    plt.xlabel("Date")
    plt.ylabel("Average PM2.5")
    plt.show()
else:
    print("Date column not available.")


In [ ]:
# Correlation heatmap
numeric_df = df.select_dtypes(include=np.number)
plt.figure(figsize=(12, 8))
sns.heatmap(numeric_df.corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Heatmap of Numerical Features")
plt.show()


In [ ]:
# Scatterplots with PM2.5
candidate_features = [col for col in ["TEMP", "PRES", "DEWP", "Iws"] if col in df.columns]

for col in candidate_features:
    plt.figure(figsize=(7, 4))
    sns.scatterplot(data=df.sample(min(3000, len(df)), random_state=42), x=col, y="pm2.5", alpha=0.6)
    plt.title(f"{col} vs PM2.5")
    plt.xlabel(col)
    plt.ylabel("PM2.5")
    plt.show()


In [ ]:
# Seasonal PM2.5 analysis
if "season_name" in df.columns:
    plt.figure(figsize=(8, 5))
    sns.boxplot(data=df, x="season_name", y="pm2.5", order=["Winter", "Spring", "Summer", "Autumn"])
    plt.title("Season-wise Distribution of PM2.5")
    plt.xlabel("Season")
    plt.ylabel("PM2.5")
    plt.show()
else:
    print("Season feature not available.")


### Key Observations

- PM2.5 values are often highly variable and may include significant outliers.  
- Environmental variables such as dew point, pressure, temperature, and wind speed may influence PM2.5 differently.  
- Seasonal patterns can reveal periods of consistently higher or lower pollution.


## 6. Pollution Risk Label Creation

To support risk analysis, PM2.5 values are grouped into pollution categories. These thresholds are simplified for this project.


In [ ]:
def pollution_risk(pm):
    if pm <= 50:
        return "Low"
    elif pm <= 100:
        return "Moderate"
    elif pm <= 150:
        return "High"
    else:
        return "Very High"

df["risk"] = df["pm2.5"].apply(pollution_risk)
df["risk"].value_counts()


In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x="risk", order=["Low", "Moderate", "High", "Very High"])
plt.title("Distribution of Pollution Risk Categories")
plt.xlabel("Risk Category")
plt.ylabel("Count")
plt.show()


## 7. Feature Selection

We use both meteorological and contextual features, depending on availability in the dataset.

- Numerical candidates: `DEWP`, `TEMP`, `PRES`, `Iws`, `Is`, `Ir`, `year`, `month`, `day`, `hour`
- Categorical candidates: `cbwd`, `season_name`, `time_of_day`


In [ ]:
target_reg = "pm2.5"
target_clf = "risk"

numeric_features = [col for col in ["DEWP", "TEMP", "PRES", "Iws", "Is", "Ir", "year", "month", "day", "hour"] if col in df.columns]
categorical_features = [col for col in ["cbwd", "season_name", "time_of_day"] if col in df.columns]

selected_features = numeric_features + categorical_features

print("Numerical features:", numeric_features)
print("Categorical features:", categorical_features)
print("Total selected features:", selected_features)


## 8. Train-Test Split and Preprocessing Pipeline

We prepare reusable preprocessing steps:

- impute missing values  
- scale numerical features  
- one-hot encode categorical variables


In [ ]:
X = df[selected_features].copy()
y_reg = df[target_reg].copy()
y_clf = df[target_clf].copy()

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X, y_reg, test_size=0.2, random_state=42
)

X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)


## 9. Regression Models for PM2.5 Forecasting

We compare:

1. **Linear Regression**  
2. **Random Forest Regressor**


In [ ]:
# Linear Regression pipeline
lr_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

lr_model.fit(X_train_reg, y_train_reg)
y_pred_lr = lr_model.predict(X_test_reg)

lr_mae = mean_absolute_error(y_test_reg, y_pred_lr)
lr_rmse = np.sqrt(mean_squared_error(y_test_reg, y_pred_lr))
lr_r2 = r2_score(y_test_reg, y_pred_lr)

print("Linear Regression Results")
print("MAE :", round(lr_mae, 3))
print("RMSE:", round(lr_rmse, 3))
print("R2  :", round(lr_r2, 3))


In [ ]:
# Random Forest Regressor pipeline
rf_reg_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=150,
        random_state=42,
        n_jobs=-1
    ))
])

rf_reg_model.fit(X_train_reg, y_train_reg)
y_pred_rf = rf_reg_model.predict(X_test_reg)

rf_mae = mean_absolute_error(y_test_reg, y_pred_rf)
rf_rmse = np.sqrt(mean_squared_error(y_test_reg, y_pred_rf))
rf_r2 = r2_score(y_test_reg, y_pred_rf)

print("Random Forest Regressor Results")
print("MAE :", round(rf_mae, 3))
print("RMSE:", round(rf_rmse, 3))
print("R2  :", round(rf_r2, 3))


In [ ]:
# Regression model comparison table
regression_results = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest Regressor"],
    "MAE": [lr_mae, rf_mae],
    "RMSE": [lr_rmse, rf_rmse],
    "R2 Score": [lr_r2, rf_r2]
}).sort_values(by="R2 Score", ascending=False)

regression_results


In [ ]:
# Actual vs predicted plot for best regression model
best_reg_name = regression_results.iloc[0]["Model"]

if best_reg_name == "Random Forest Regressor":
    best_pred = y_pred_rf
else:
    best_pred = y_pred_lr

plt.figure(figsize=(7, 7))
plt.scatter(y_test_reg, best_pred, alpha=0.5)
plt.xlabel("Actual PM2.5")
plt.ylabel("Predicted PM2.5")
plt.title(f"Actual vs Predicted PM2.5 ({best_reg_name})")
plt.plot([y_test_reg.min(), y_test_reg.max()], [y_test_reg.min(), y_test_reg.max()])
plt.show()


### Regression Interpretation

The model with lower **MAE** and **RMSE**, and higher **R²**, performs better.  
Random Forest often performs better than Linear Regression when the relationship between variables is nonlinear.


## 10. Classification Models for Pollution Risk Analysis

We compare:

1. **Logistic Regression**  
2. **Random Forest Classifier**


In [ ]:
# Logistic Regression pipeline
log_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=2000))
])

log_model.fit(X_train_clf, y_train_clf)
y_pred_log = log_model.predict(X_test_clf)

log_acc = accuracy_score(y_test_clf, y_pred_log)
log_precision = precision_score(y_test_clf, y_pred_log, average="weighted")
log_recall = recall_score(y_test_clf, y_pred_log, average="weighted")
log_f1 = f1_score(y_test_clf, y_pred_log, average="weighted")

print("Logistic Regression Results")
print("Accuracy :", round(log_acc, 3))
print("Precision:", round(log_precision, 3))
print("Recall   :", round(log_recall, 3))
print("F1-score :", round(log_f1, 3))


In [ ]:
# Random Forest Classifier pipeline
rf_clf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=150,
        random_state=42,
        n_jobs=-1
    ))
])

rf_clf_model.fit(X_train_clf, y_train_clf)
y_pred_clf = rf_clf_model.predict(X_test_clf)

rf_acc = accuracy_score(y_test_clf, y_pred_clf)
rf_precision = precision_score(y_test_clf, y_pred_clf, average="weighted")
rf_recall = recall_score(y_test_clf, y_pred_clf, average="weighted")
rf_f1 = f1_score(y_test_clf, y_pred_clf, average="weighted")

print("Random Forest Classifier Results")
print("Accuracy :", round(rf_acc, 3))
print("Precision:", round(rf_precision, 3))
print("Recall   :", round(rf_recall, 3))
print("F1-score :", round(rf_f1, 3))


In [ ]:
# Classification model comparison table
classification_results = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest Classifier"],
    "Accuracy": [log_acc, rf_acc],
    "Precision": [log_precision, rf_precision],
    "Recall": [log_recall, rf_recall],
    "F1 Score": [log_f1, rf_f1]
}).sort_values(by="F1 Score", ascending=False)

classification_results


In [ ]:
# Detailed classification report for best classifier
best_clf_name = classification_results.iloc[0]["Model"]

if best_clf_name == "Random Forest Classifier":
    best_y_pred_clf = y_pred_clf
else:
    best_y_pred_clf = y_pred_log

print(f"Best Classification Model: {best_clf_name}\n")
print(classification_report(y_test_clf, best_y_pred_clf))


In [ ]:
# Confusion matrix for best classifier
cm = confusion_matrix(y_test_clf, best_y_pred_clf, labels=sorted(y_test_clf.unique()))

plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=sorted(y_test_clf.unique()),
            yticklabels=sorted(y_test_clf.unique()))
plt.title(f"Confusion Matrix ({best_clf_name})")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()


### Classification Interpretation

A higher **accuracy** and **F1-score** indicates stronger classification performance.  
The confusion matrix helps identify which pollution categories are easier or harder for the model to classify.


## 11. Sample Predictions

This section shows how the best models can be used to generate predictions.


In [ ]:
sample_rows = X_test_reg.head(10).copy()

# Regression predictions
sample_rows["Actual_PM2.5"] = y_test_reg.head(10).values
sample_rows["Predicted_PM2.5_LR"] = lr_model.predict(X_test_reg.head(10))
sample_rows["Predicted_PM2.5_RF"] = rf_reg_model.predict(X_test_reg.head(10))

sample_rows


In [ ]:
sample_class = X_test_clf.head(10).copy()
sample_class["Actual_Risk"] = y_test_clf.head(10).values
sample_class["Predicted_Risk_Logistic"] = log_model.predict(X_test_clf.head(10))
sample_class["Predicted_Risk_RF"] = rf_clf_model.predict(X_test_clf.head(10))

sample_class


## 12. Key Findings

- Machine learning can be used effectively for **PM2.5 prediction** and **pollution risk classification**.  
- Environmental and weather-related variables provide meaningful signals for air quality forecasting.  
- Comparing multiple models makes the project stronger and helps identify the most effective approach.


## 13. Conclusion

This project successfully explored air quality forecasting and pollution risk analysis using the Beijing PM2.5 dataset. After cleaning and analyzing the data, both regression and classification models were built to understand pollution behavior.

The regression task estimated PM2.5 concentration, while the classification task translated those values into meaningful pollution risk levels. Comparing multiple models provided a more reliable evaluation of performance. Overall, the project demonstrates how machine learning can support environmental monitoring, decision-making, and public health awareness.


## 14. Future Improvements

Possible extensions for this project include:

- using advanced models such as **XGBoost**, **LightGBM**, or **LSTM**  
- including additional pollutants and external weather data  
- deploying the model in a **dashboard** for interactive prediction  
- adding real-time forecasting and alert systems


## 15. Final Note

If you get a file-not-found error, make sure your dataset CSV is in the same folder as this notebook, or update the file path in:

```python
DATA_PATH = "PRSA_data_2010.1.1-2014.12.31.csv"
```
